In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import json
from baseline_model import BaselineModel
from kitchen_simulator import KitchenSimulator
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
# Load data
df = pd.read_csv('../data/synthetic_orders.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

with open('../data/merchant_profiles.json') as f:
    merchants = {m['merchant_id']: m for m in json.load(f)}

print(f"Loaded {len(df)} orders")

In [ ]:
# Run baseline simulation
baseline = BaselineModel()
results = []

for idx, order in df.iterrows():
    merchant = merchants[order['merchant_id']]
    result = baseline.simulate_order(order, merchant)
    results.append(result)
    
    if idx % 1000 == 0:
        print(f"Processed {idx} orders...")

baseline_results = pd.DataFrame(results)
baseline_results.to_csv('../results/baseline_results.csv', index=False)
print("Saved baseline_results.csv")

In [ ]:
# Calculate metrics
metrics = {
    'avg_rider_wait': baseline_results['rider_wait'].mean(),
    'p50_eta_error': baseline_results['eta_error'].quantile(0.5),
    'p90_eta_error': baseline_results['eta_error'].quantile(0.9),
    'delay_rate': baseline_results['is_delayed'].mean(),
    'avg_idle_time': baseline_results['rider_wait'].mean() + 4
}

print("\n=== BASELINE METRICS ===")
for key, value in metrics.items():
    print(f"{key}: {value:.2f}")

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Wait time distribution
axes[0,0].hist(baseline_results['rider_wait'], bins=50, edgecolor='black')
axes[0,0].axvline(metrics['avg_rider_wait'], color='r', linestyle='--', label='Mean')
axes[0,0].set_title('Rider Wait Time Distribution')
axes[0,0].set_xlabel('Minutes')
axes[0,0].legend()

# ETA error
axes[0,1].hist(baseline_results['eta_error'], bins=50, edgecolor='black')
axes[0,1].axvline(metrics['p50_eta_error'], color='orange', linestyle='--', label='P50')
axes[0,1].axvline(metrics['p90_eta_error'], color='r', linestyle='--', label='P90')
axes[0,1].set_title('ETA Prediction Error')
axes[0,1].set_xlabel('Minutes')
axes[0,1].legend()

# Predicted vs True
axes[1,0].scatter(baseline_results['true_kpt'], baseline_results['predicted_kpt'], alpha=0.3)
axes[1,0].plot([0, 60], [0, 60], 'r--', label='Perfect prediction')
axes[1,0].set_title('Predicted vs True KPT')
axes[1,0].set_xlabel('True KPT (min)')
axes[1,0].set_ylabel('Predicted KPT (min)')
axes[1,0].legend()

# Delay rate pie
delay_counts = baseline_results['is_delayed'].value_counts()
axes[1,1].pie(delay_counts, labels=['On Time', 'Delayed'], autopct='%1.1f%%')
axes[1,1].set_title('Order Delay Rate')

plt.tight_layout()
plt.savefig('../results/baseline_analysis.png', dpi=150)
plt.show()